<a href="https://colab.research.google.com/github/Sreenithy22/Machine-Learning-Projects/blob/main/Pos/Neg%20Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch sklearn accelerate

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

In [2]:
from transformers import BertTokenizer,BertForSequenceClassification
from transformers import Trainer,TrainingArguments
from datasets import Dataset

In [3]:
df=pd.read_csv("/content/Flipkart_reviews - Copy.csv")
print(df.head())
print("Dataset Size:",len(df))

                                        product_name  product_price  Rate  \
0  Candes 12 L Room/Personal Air Cooler??????(Whi...           3999     5   
1  Candes 12 L Room/Personal Air Cooler??????(Whi...           3999     5   
2  Candes 12 L Room/Personal Air Cooler??????(Whi...           3999     3   
3  Candes 12 L Room/Personal Air Cooler??????(Whi...           3999     1   
4  Candes 12 L Room/Personal Air Cooler??????(Whi...           3999     3   

            Review                                            Summary  \
0           super!  great cooler excellent air flow and for this p...   
1          awesome              best budget 2 fit cooler nice cooling   
2             fair  the quality is good but the power of air is de...   
3  useless product                  very bad product its a only a fan   
4             fair                                      ok ok product   

  Sentiment  
0  positive  
1  positive  
2  positive  
3  negative  
4   neutral  
Dataset Size: 

In [4]:
df["Review"]=df["Review"].fillna("")
df["Summary"]=df["Summary"].fillna("")
df["text"]=df["Review"]+" "+df["Summary"]
#Remove empty rows
df=df[df["text"].str.strip()!=""]
print("After Cleaning:",len(df))

After Cleaning: 10005


In [5]:
label_encoder=LabelEncoder()
df["label"]=label_encoder.fit_transform(df["Sentiment"])
print("Classes:",label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [6]:
#train-test Split
train_df,test_df=train_test_split(
    df[["text","label"]],
    test_size=0.2,
    random_state=42
)

In [7]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
print(train_dataset.column_names)

['text', 'label', '__index_level_0__']


In [8]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
def tokenize_function(example):
  texts = [str(t) for t in example["text"]]
  return tokenizer(texts, padding="max_length", truncation=True, max_length=128)

In [10]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/8004 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

In [11]:
train_dataset = train_dataset.remove_columns(["text","__index_level_0__"])
test_dataset = test_dataset.remove_columns(["text","__index_level_0__"])

In [12]:
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [13]:
model=BertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=3)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
training_args=TrainingArguments(
    output_dir="./resulrs",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

In [15]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.285303,0.168516
2,0.181815,0.105808
3,0.107886,0.064132


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3003, training_loss=0.20662069478380185, metrics={'train_runtime': 745.8386, 'train_samples_per_second': 32.195, 'train_steps_per_second': 4.026, 'total_flos': 1579469846621184.0, 'train_loss': 0.20662069478380185, 'epoch': 3.0})

In [17]:
results=trainer.evaluate()
print(results)

{'eval_loss': 0.0641324520111084, 'eval_runtime': 55.1663, 'eval_samples_per_second': 145.089, 'eval_steps_per_second': 18.145, 'epoch': 3.0}


In [18]:
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(-1)
print(classification_report(test_df["label"], pred_labels))


              precision    recall  f1-score   support

           0       0.83      0.89      0.86       233
           1       0.61      0.32      0.42        85
           2       0.96      0.98      0.97      1683

    accuracy                           0.94      2001
   macro avg       0.80      0.73      0.75      2001
weighted avg       0.93      0.94      0.93      2001



In [29]:
def predict_sentiment(text):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  inputs = tokenizer(
      text,
      return_tensors="pt",
      truncation=True,
      padding=True
  )
  inputs = {key: value.to(device) for
            key, value in inputs.items()}
  with torch.no_grad():
      outputs = model(**inputs)

  prediction = torch.argmax(outputs.logits,dim=1).item()
  return label_encoder.inverse_transform([prediction])[0]

In [26]:
print(predict_sentiment("Not worth for the money"))
print(predict_sentiment("Quality inconsistency: Beige top has different material and patterns compared to others."))

negative
neutral


In [27]:
model.save_pretrained("bert_sentiment_model")
tokenizer.save_pretrained("bert_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_sentiment_model/tokenizer_config.json',
 'bert_sentiment_model/tokenizer.json')